# 03 — What does Semantica add?

Think of Semantica ContextGraph as a **decision card box with an index**.

Notebook 2 made one run readable. Notebook 3 asks what we can do after many runs have
been stored as native Semantica Decisions. We keep only four useful questions:

1. What is on one decision card?
2. Can we find similar cards?
3. Did the same question receive different answers?
4. If a policy changes, which old cards must a human re-check?

A card never replaces the raw trace. Its provenance pointer is the receipt number that
lets us go back to the original run.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import os
import shutil

from IPython.display import Markdown, display
from acr.mvp.ledger import PROJECTION_SCHEMA, SemanticaLedger

START = Path.cwd().resolve()
ROOT = START if (START / "pyproject.toml").is_file() else START.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"

source_paths = [
    ROOT / "runs/policy-experiment-20260827/experiment-ledger.json",
    ROOT / "runs/notebook-live-20260827/ledger.json",
    ROOT / "runs/postdoc-study/ledger.json",
]
source_paths = [path for path in source_paths if path.is_file()]
assert source_paths, "Run Notebook 1 first"

output_dir = ROOT / "runs/postdoc-notebook-output"
output_dir.mkdir(parents=True, exist_ok=True)
version = PROJECTION_SCHEMA.rsplit(".", 1)[-1]
LEDGER_PATH = Path(os.environ.get(
    "ACR_INTELLIGENCE_LEDGER", output_dir / f"readable-context-graph-{version}.json"
))
ledger = SemanticaLedger(LEDGER_PATH)

selected_members = {}
for source_path in source_paths:
    source = SemanticaLedger(source_path)
    runs = sorted({
        str((node.get("metadata") or {}).get("run_id"))
        for node in source.graph.find_nodes(node_type="decision")
    })
    for run_id in runs:
        available = source.available_analyses(run_id)
        if not available:
            continue
        analysis_id = source.selected_analysis(run_id) or sorted(available)[0]
        selected_members.setdefault(run_id, (source, analysis_id))

# A v3 Decision projection changes Semantica decision ids. Keep the sealed v2
# run-local provenance immutable by staging artifact pointers beside this tutorial
# graph; the original run directories are read-only inputs.
staging_root = output_dir / f"projection-staging-{version}"
for run_id, (source, analysis_id) in selected_members.items():
    artifact = source.load_analysis_artifact(run_id, analysis_id)
    original_ref = Path(artifact["artifact_ref"])
    original_run = original_ref.parent.parent
    staged_run = staging_root / run_id
    staged_analyses = staged_run / "analyses"
    staged_analyses.mkdir(parents=True, exist_ok=True)
    for name in (
        "task_presentation.json", "runner_meta.json", "result.json",
        "trace.jsonl", "trace_manifest.json"
    ):
        source_file = original_run / name
        if source_file.is_file():
            shutil.copy2(source_file, staged_run / name)
    staged_ref = staged_analyses / original_ref.name
    artifact = json.loads(json.dumps(artifact))
    artifact["artifact_ref"] = str(staged_ref)
    staged_ref.write_text(
        json.dumps(artifact, ensure_ascii=False, indent=2) + "\n"
    )
    ledger.project_analysis(artifact)

def case_name(run_id):
    parts = str(run_id).split("_", 2)
    return parts[1] if len(parts) > 2 else "case"

def short_model(value):
    return "Terra" if "terra" in str(value).lower() else "Luna"

def one_line(value, limit=125):
    text = " ".join(str(value or "").split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def table(rows, columns, limit=160):
    def clean(value):
        return one_line(value, limit).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(clean(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

def episode_for_node(node):
    meta = node.get("metadata") or {}
    artifact = ledger.load_analysis_artifact(meta["run_id"], meta["analysis_id"])
    episode = next(
        row for row in artifact["episodes"]
        if row["episode_id"] == meta["acr_episode_id"]
    )
    return artifact, episode

decisions = ledger.graph.find_nodes(node_type="decision")
display(Markdown(
    f"**Loaded:** {len(selected_members)} runs → {len(decisions)} decision cards."
))


## 1. One decision card

Semantica's native unit is simple: `category`, `scenario`, `reasoning`, `outcome`, and
`confidence`. The main fields are human-readable and de-identified. Exact dates, note
locators, and field-level provenance remain behind the card in the audit record.


In [ ]:
query_node = next(
    node for node in decisions
    if (node.get("metadata") or {}).get("decision_subject") == "evidence_item"
    and (node.get("metadata") or {}).get("outcome") == "MERELY_MENTIONS"
    and "reports atypical cells suspicious" in str(
        (node.get("metadata") or {}).get("scenario", "")
    ).lower()
)
query_meta = query_node["metadata"]
decision_card = {
    "category": query_meta["category"],
    "scenario": query_meta["scenario"],
    "reasoning": query_meta["reasoning"],
    "outcome": query_meta["outcome"],
    "confidence": query_meta["confidence"],
}
display(Markdown(table(
    [{"field": key.title(), "value": value}
     for key, value in decision_card.items()],
    [("field", "Decision field"), ("value", "Recorded value")],
    limit=600,
)))
display(Markdown(
    "**How to read confidence:** it measures reconstruction stability across passes, "
    "not clinical correctness. The card is still something a human must judge."
))


This is the chart-review equivalent of Semantica's vendor-selection example:

```python
graph.record_decision(
    category="standing",
    scenario="Can suspicious cytology alone establish the diagnosis date?",
    reasoning="The report is ambiguous and recommends confirmatory biopsy.",
    outcome="MERELY_MENTIONS",
    confidence=0.88,  # reconstruction stability, not correctness
)
```

## 2. Find similar decisions

Similarity is useful because a reviewer can inspect a small set of prior judgments
instead of searching every trace. It proposes comparison candidates; it does not say
that either judgment is correct.

**Semantica—not ACR—calculates the similarity.** First we ask its native API for
neighbours:

```python
similar = graph.find_similar_decisions(
    "Can suspicious cytology alone establish the diagnosis date?",
    category="standing",
    max_results=5,
)
```

Semantica's raw list can contain the query card itself and cards from the same run.
We show that raw result first. Then ACR turns it into an audit list by removing the
query, limiting comparison to other runs, and marking exact decision points. It never
recalculates Semantica's score.


In [ ]:
native_similar = ledger.graph.find_similar_decisions(
    query_meta["scenario"],
    category=query_meta["category"],
    max_results=5,
    min_similarity=0.0,
)
native_rows = []
for candidate in native_similar:
    node = candidate["decision"]
    meta = node.get("metadata") or {}
    if str(node.get("id") or "") == str(query_node.get("id") or ""):
        scope = "Query card itself"
    elif meta.get("run_id") == query_meta.get("run_id"):
        scope = "Another card in same run"
    else:
        scope = "Card from another run"
    native_rows.append({
        "scope": scope,
        "scenario": node.get("scenario") or meta.get("scenario"),
        "outcome": node.get("outcome") or meta.get("outcome"),
        "similarity": f"{float(candidate.get('similarity') or 0):.2f}",
    })

display(Markdown("**A. Raw neighbours returned by Semantica**"))
display(Markdown(table(native_rows, [
    ("scope", "What kind of match?"), ("scenario", "Comparable question"),
    ("outcome", "Outcome"), ("similarity", "Semantica score"),
])))
display(Markdown(
    "The query itself appears first. Its score need not be 1.0 because this is a "
    "composite retrieval score, not an identity check or probability."
))

query_episode = query_meta["acr_episode_id"]
audit_candidates = ledger.similar_candidates(
    query_episode, max_results=6, min_similarity=0.05, cross_run_only=True
)
seen_runs = set()
similar_rows = []
for candidate in audit_candidates["candidates"]:
    node = candidate["decision"]
    meta = node.get("metadata") or {}
    # Presentation-only thinning: one example card per run.
    if meta.get("run_id") in seen_runs:
        continue
    seen_runs.add(meta.get("run_id"))
    candidate_artifact, _ = episode_for_node(node)
    arm = candidate_artifact.get("task_arm")
    similar_rows.append({
        "context": (
            f"{case_name(meta.get('run_id'))} · "
            f"{short_model(candidate_artifact.get('review_model'))} · "
            f"{'Task + policy' if arm == 'policy_bundle' else 'Task only'}"
        ),
        "scenario": node.get("scenario") or meta.get("scenario"),
        "outcome": node.get("outcome") or meta.get("outcome"),
        "similarity": f"{float(candidate.get('similarity') or 0):.2f}",
    })
    if len(similar_rows) == 3:
        break

display(Markdown(
    f"**B. Cross-run audit list prepared by ACR**\n\n"
    f"**Query card:** {query_meta['scenario']}"
))
display(Markdown(table(similar_rows, [
    ("context", "Run context"), ("scenario", "Comparable question"),
    ("outcome", "Outcome"),
    ("similarity", "Similarity")
])))
display(Markdown(
    "**What happened?** Semantica retrieved the neighbours and supplied every "
    "similarity score. ACR removed self/same-run matches and added audit context; this "
    "table then shows one card per run for readability. A score closer to 1 means "
    "more alike; it is not an accuracy score."
))


## 3. Find the same question with different answers

“Similar” is broad. A stronger audit signal is **the same case, the same evidence note,
and the same atomic question**, but a different outcome.

This comparison still begins with Semantica's native `find_similar_decisions()`
candidates. ACR then checks exact evidence identity and keeps pairs whose outcomes
differ; it does not provide a second similarity engine.

The stored runs contain exactly that pattern for one `SYN0001` oncology note. In the
task-only arm, no clinical policy was provided, so the models had to use their own
judgment.


In [ ]:
cohort = [
    {"run_id": run_id, "analysis_id": analysis_id}
    for run_id, (_, analysis_id) in selected_members.items()
    if "task_only" in run_id
]
report = ledger.find_divergent_decision_points(cohort, min_similarity=0.05)
exact = [row for row in report["divergences"] if row.get("same_decision_point")]
assert exact, "Expected a same-evidence task-only disagreement"
disagreement = max(exact, key=lambda row: len(row.get("members") or []))

disagreement_rows = []
for member in disagreement["members"]:
    node = ledger.graph.find_node(member["decision_id"])
    meta = node.get("metadata") or {}
    disagreement_rows.append({
        "case": case_name(member["run_id"]),
        "model": short_model(member["review_model"]),
        "answer": member["outcome"],
        "reason": meta.get("reasoning"),
        "policy": "None — task only",
    })
display(Markdown(table(disagreement_rows, [
    ("case", "Case"), ("model", "Model"), ("answer", "Answer to same question"),
    ("reason", "Recorded reasoning"), ("policy", "Policy grounding")
], limit=280)))
display(Markdown(
    "**So what?** This is a guideline-gap candidate: the same evidence received "
    "different standings when the task supplied no rule. A human should decide the "
    "desired rule, then add it to the guideline."
))


## 4. If a policy changes, what must be re-checked?

Suppose we tighten this policy:

> If a physician diagnosis predates tissue confirmation, use the earlier physician
> date; later tissue confirmation does not reset an already established diagnosis.

Semantica stores versioned Policy nodes and direct `APPLIED_POLICY` links. That lets us
retrieve the historical decisions that cited this policy.


In [ ]:
policy_id = (
    "STORE.390.date_of_initial_diagnosis."
    "conflict.physician_statement_predating_tissue"
)
policy_nodes = [
    node for node in ledger.graph.find_nodes(node_type="Policy")
    if (node.get("metadata") or {}).get("policy_id") == policy_id
]
assert policy_nodes, "The detailed runs contain no matching policy"
old = policy_nodes[0]["metadata"]
revision = ledger.register_policy_revision(
    policy_id,
    from_version=old["version"],
    rules={
        "plain_language": (
            "Use an earlier physician diagnosis only when it explicitly identifies "
            "the target tumour; later tissue confirmation does not reset that date."
        )
    },
    change_reason="Require an explicit target-tumour statement.",
)
impact = ledger.affected_by_policy_change(
    policy_id,
    from_version=old["version"],
    to_version=revision["version"],
)
functions = sorted({
    function
    for case in impact["affected_cases"]
    for function in case["decision_functions"]
})
friendly = {
    "where_to_look": "Search / choose notes",
    "standing": "Judge evidence",
    "which_wins": "Resolve conflicts",
    "enough": "Decide whether to stop",
    "what_to_answer": "Choose the final answer",
}
affected_runs = {row["run_id"] for row in impact["affected_cases"]}
display(Markdown(
    f"- **Historical runs to revisit:** {len(affected_runs)}\n"
    f"- **Directly bound decisions:** {len(impact['affected_decisions'])}\n"
    f"- **Parts of the review involved:** "
    f"{', '.join(friendly.get(item, item) for item in functions)}\n"
    "- **Meaning:** this is a re-audit queue, not a prediction that every answer changes."
))


## The whole point

| Without the ContextGraph | With the ContextGraph |
|---|---|
| Read one trace at a time | Retrieve a small set of comparable Decision cards |
| Notice only final-answer differences | Localize disagreement to one evidence judgment |
| Guess which old runs a rule touched | Ask Semantica for directly policy-bound decisions |
| Trust a summary | Follow the card's provenance back to the raw trace |

**Use the graph to route human attention. Use provenance and raw trace to decide what
really happened. Use a qualified reviewer to decide what is clinically correct.**
